In [3]:
# Importing dependencies
import duckdb # instance database for sql
import pandas as pd # data handler
import json

In [4]:
!git clone https://github.com/Bcliffwm/awesome_database_project.git

Cloning into 'awesome_database_project'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 13 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 1.33 MiB | 3.63 MiB/s, done.
Resolving deltas: 100% (3/3), done.


EDA Step and Loading Data Into Environment

In [5]:
# Reading data from its source
my_df = pd.read_json('/content/awesome_database_project/US_STATE_recipes.json', orient='index').rename(columns={'Contient': 'Continent'})
my_df.head(5)

,Continent,Country_State,cuisine,title,URL,rating,total_time,prep_time,cook_time,description,ingredients,instructions,nutrients,serves
0,Africa,NaN,Missouri,Ground Beef and Cabbage,https://www.allrecipes.com/recipe/229324/groun...,4.5,60.0,15.0,45.0,This ground beef and cabbage recipe combines l...,"[1 large head cabbage, finely chopped, 1 (14.5...","[Place cabbage, tomatoes with juice, onion, It...","{'calories': '228 kcal', 'carbohydrateContent'...",6 servings
1,Africa,NaN,Missouri,Old Fashioned Peach Cobbler,https://www.allrecipes.com/recipe/19897/old-fa...,4.6,130.0,30.0,70.0,This old-fashioned peach cobbler recipe featur...,"[2.5 cups all-purpose flour, 4 tablespoons whi...","[Make crust: Sift together flour, 3 tablespoon...","{'calories': '338 kcal', 'carbohydrateContent'...",18 servings
2,Africa,NaN,Missouri,St. Louis Toasted Ravioli,https://www.allrecipes.com/recipe/16907/st-lou...,4.5,25.0,15.0,10.0,Toasted ravioli traces its roots to St. Louis....,"[1 (16 ounce) jar marinara sauce, 1 large egg,...",[Heat marinara sauce in a saucepan over medium...,"{'calories': '374 kcal', 'carbohydrateContent'...",6 servings
3,Africa,NaN,Missouri,Amish Friendship Bread Starter,https://www.allrecipes.com/recipe/7063/amish-f...,4.7,14440.0,30.0,NaN,Amish friendship bread starter is made with ye...,"[1 (.25 ounce) package active dry yeast, 0.25 ...",[Dissolve yeast in warm water in a small bowl;...,"{'calories': '34 kcal', 'carbohydrateContent':...",120 servings
4,Africa,NaN,Missouri,Simple Fried Morel Mushrooms,https://www.allrecipes.com/recipe/220833/simpl...,4.7,30.0,20.0,10.0,"The rich, meaty flavor of fresh morel mushroom...",[1 pound fresh morel mushrooms - dirt gently b...,[Place halved morel mushrooms in a large bowl;...,"{'calories': '185 kcal', 'carbohydrateContent'...",4 servings


In [6]:
# Profiling global statistics of the dataset we are working with
my_df.describe()

,Country_State,rating,total_time,prep_time,cook_time
count,0.0,3530.000000,3699.000000,3488.000000,2832.000000
mean,NaN,4.469178,164.361719,19.339450,54.931497
std,NaN,0.377538,962.533936,41.798683,111.301829
min,NaN,1.000000,0.000000,1.000000,1.000000
25%,NaN,4.300000,25.000000,10.000000,15.000000
50%,NaN,4.500000,55.000000,15.000000,30.000000
75%,NaN,4.700000,100.000000,20.000000,50.000000
max,NaN,5.000000,30280.000000,1800.000000,2701.000000


In [7]:
# Notice Country_State is entirely NaN - dropping any null values from the dataset
# Dropping Country_State from the dataframe
my_df = my_df.drop(columns=['Country_State'])

my_df.dropna(axis=0, inplace=True)

In [8]:
# Understanding the dataframe's structure to form schema
my_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2682 entries, 0 to 3762
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Continent     2682 non-null   object 
 1   cuisine       2682 non-null   object 
 2   title         2682 non-null   object 
 3   URL           2682 non-null   object 
 4   rating        2682 non-null   float64
 5   total_time    2682 non-null   float64
 6   prep_time     2682 non-null   float64
 7   cook_time     2682 non-null   float64
 8   description   2682 non-null   object 
 9   ingredients   2682 non-null   object 
 10  instructions  2682 non-null   object 
 11  nutrients     2682 non-null   object 
 12  serves        2682 non-null   object 
dtypes: float64(4), object(9)
memory usage: 293.3+ KB


In [9]:
# jsonifying the string column - nutrients
normalized_data = [] # list to hold jsonified objects

# iterating through the column
for row in my_df['nutrients']:
  normalized_data.append(pd.json_normalize(row)) # using pandas to jsonify each row's entry into a dataframe

nutrients_df = pd.concat(normalized_data, ignore_index=True) # concatenating each smaller df into one comprehense dataframe

In [10]:
# Dropping redundant or useless columns from dataframes
nutrients_df.drop(columns=['servingSize'], inplace=True)
my_df.drop(columns=(['ingredients', 'instructions', 'nutrients', 'cuisine']), inplace=True)

In [11]:
# Converting each record from column - serves into an int data type
my_df['serves'] = [int(entry.split(' ', 1)[0]) for entry in my_df['serves']]

In [12]:
my_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2682 entries, 0 to 3762
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Continent    2682 non-null   object 
 1   title        2682 non-null   object 
 2   URL          2682 non-null   object 
 3   rating       2682 non-null   float64
 4   total_time   2682 non-null   float64
 5   prep_time    2682 non-null   float64
 6   cook_time    2682 non-null   float64
 7   description  2682 non-null   object 
 8   serves       2682 non-null   int64  
dtypes: float64(4), int64(1), object(4)
memory usage: 209.5+ KB


In [13]:
# Creating index cols for joining the two tables together
my_df['id'] = my_df.index #primary key for our recipe database
nutrients_df['recipe_id'] = nutrients_df.index # foreign key to id column of my_df table

In [14]:
nutrients_df[0:2]

,calories,carbohydrateContent,cholesterolContent,fiberContent,proteinContent,saturatedFatContent,sodiumContent,sugarContent,fatContent,unsaturatedFatContent,recipe_id
0,228 kcal,18 g,50 mg,7 g,18 g,4 g,191 mg,10 g,10 g,0 g,0
1,338 kcal,44 g,26 mg,1 g,2 g,7 g,177 mg,30 g,18 g,0 g,1


In [15]:
# Creating an ordered column based on desired table structure for reading in database
recipe_df = my_df.loc[:, ['id', 'title', 'description', 'Continent', 'rating', 'prep_time', 'cook_time', 'serves']]

In [16]:
nutrients_df.rename(columns={
    'carbohydrateContent': 'carbs',
    'cholesterolContent': 'cholesterol',
    'fiberContent': 'fiber',
    'proteinContent': 'protein',
    'saturatedFatContent': 'sat_fats',
    'sodiumContent': 'sodium',
    'sugarContent': 'sugar',
    'unsaturatedFatContent': 'unsat_fats'
}, inplace=True)

nutrients_df = nutrients_df.loc[:, ['recipe_id', 'calories', 'carbs', 'cholesterol', 'fiber', 'protein', 'sat_fats', 'sodium', 'sugar', 'unsat_fats']]

Initializing Instance Database for testing

In [17]:
recipe_df.keys()

Index(['id', 'title', 'description', 'Continent', 'rating', 'prep_time',
       'cook_time', 'serves'],
      dtype='object')

In [18]:
# Initializing the instance database
con = duckdb.connect(database=':memory')

In [19]:
con.execute("""CREATE TABLE recipe (id INTEGER,
  title VARCHAR(30),
  description VARCHAR(1000),
  Continent VARCHAR(30),
  rating DOUBLE,
  prep_time DOUBLE,
  cook_time DOUBLE,
  serves INTEGER
)""")

In [20]:
con.execute("""INSERT INTO recipe (SELECT * FROM recipe_df)""")

In [21]:
nutrients_df.keys()

Index(['recipe_id', 'calories', 'carbs', 'cholesterol', 'fiber', 'protein',
       'sat_fats', 'sodium', 'sugar', 'unsat_fats'],
      dtype='object')

In [22]:
con.execute("""SELECT * FROM recipe""").fetchone()

(0,
 'Ground Beef and Cabbage',
 'This ground beef and cabbage recipe combines lean ground beef, cabbage, onion, and tomatoes with simple seasonings for a hearty and comforting dish.',
 'Africa',
 4.5,
 15.0,
 45.0,
 6)

In [23]:
con.execute("""DROP TABLE IF EXISTS nutrients;
CREATE TABLE nutrients (
  recipe_id INTEGER,
  calories VARCHAR(10),
  carbs VARCHAR(10),
  cholesterol VARCHAR(10),
  fiber VARCHAR(10),
  protein VARCHAR(10),
  sat_fats VARCHAR(10),
  sodium VARCHAR(10),
  sugar VARCHAR(10),
  unsat_fats VARCHAR(10)
)""")

In [24]:
con.execute("""INSERT INTO nutrients(SELECT * FROM nutrients_df)""")

In [25]:
con.execute("""SELECT * FROM nutrients WHERE left(unsat_fats, 1) = '0'
LIMIT 3""").fetchdf()

,recipe_id,calories,carbs,cholesterol,fiber,protein,sat_fats,sodium,sugar,unsat_fats
0,0,228 kcal,18 g,50 mg,7 g,18 g,4 g,191 mg,10 g,0 g
1,1,338 kcal,44 g,26 mg,1 g,2 g,7 g,177 mg,30 g,0 g
2,2,374 kcal,39 g,57 mg,4 g,11 g,5 g,886 mg,9 g,0 g


In [26]:
# Perform LEFT OUTER JOIN between recipe and nutrients tables
joined_df = con.execute("""
SELECT *
FROM recipe r
LEFT OUTER JOIN nutrients n
ON r.id = n.recipe_id
""").fetchdf()

In [27]:
# Preview result
print(joined_df.head())

   id                         title  \
0   0       Ground Beef and Cabbage   
1   1   Old Fashioned Peach Cobbler   
2   2     St. Louis Toasted Ravioli   
3   4  Simple Fried Morel Mushrooms   
4   5   Big Al's K.C. Bar-B-Q Sauce   

                                         description Continent  rating  \
0  This ground beef and cabbage recipe combines l...    Africa     4.5   
1  This old-fashioned peach cobbler recipe featur...    Africa     4.6   
2  Toasted ravioli traces its roots to St. Louis....    Africa     4.5   
3  The rich, meaty flavor of fresh morel mushroom...    Africa     4.7   
4  This is a Kansas City-style BBQ sauce recipe s...    Africa     4.8   

   prep_time  cook_time  serves  recipe_id  calories carbs cholesterol fiber  \
0       15.0       45.0       6          0  228 kcal  18 g       50 mg   7 g   
1       30.0       70.0      18          1  338 kcal  44 g       26 mg   1 g   
2       15.0       10.0       6          2  374 kcal  39 g       57 mg   4 g   
